# mva_05-数量化Ⅰ類

---

## **要点まとめ**

**Part Ⅱ：誤差の最小化**

「数値データ」から「数値データ」を予測する線形重回帰分析を学びました。今回は、その応用として、「質的データ（カテゴリ）」から「数値データ」を予測する 数量化Ⅰ類（Quantification Theory Type I） を学習します。

### **1. 数量化Ⅰ類とは**

説明変数が「アンケートの回答（好き/嫌い）」や「天気（晴れ/雨）」のような 質的データ（カテゴリ） である場合に用いられる回帰分析の手法です。
数学的には、質的データを **ダミー変数（Dummy Variables）** と呼ばれる $0$ または $1$ の数値に変換することで、通常の重回帰分析と同じ枠組み（誤差の最小化）で解くことができます。
機械学習の分野では、この $0/1$ への変換処理を **One-Hot Encoding（ワンホットエンコーディング）** と呼びます。

データを行列形式で整理すると、以下のようになります。

| | 目的変数| 説明変数1 (0/1) | ... | 説明変数d (0/1)  |
|---|:---:|:---:|:---:|:---:|
| **サンプル1** | $y_{1}$ | $x_{11}$  | ... | $x_{1d}$ |
| **サンプル2** | $y_{2}$ | $x_{21}$ | ... | $x_{2d}$ |
| ... | ... | ... | ... | ... |
| **サンプルn** | $y_{n}$ | $x_{n1}$ | ... | $x_{nd}$ |

<br>

### **2. 数理モデルと最適化**

数量化Ⅰ類は、入力データがダミー変数（0/1）であることを除けば、数理的には通常の重回帰分析と全く同じモデルを使用します。

#### **数理モデル**

ある変数 $y$ （目的変数）の変化を、複数のダミー変数 $x_1, x_2, \dots, x_d$ （説明変数）の線形結合（足し合わせ）で説明・予測するモデルです。個々のデータサンプル $i$ について、以下の関係を仮定します。

$$y_i = w_0 + w_1 x_{i1} + w_2 x_{i2} + \dots + w_d x_{id} + \epsilon_i$$

ここで、$w$ は カテゴリウェイト（偏回帰係数）、$\epsilon$ は 誤差（ノイズ） です。
これをすべてのデータ（$n$ 個）についてまとめると、行列形式でシンプルに表現できます。

$$\mathbf{y} = \mathbf{X}\mathbf{w} + \mathbf{\epsilon}$$

- $\mathbf{y}$: 目的変数ベクトル ($n \times 1$)
- $\mathbf{X}$: デザイン行列 ($n \times (d+1)$)。ダミー変数の列に加え、先頭列にバイアス項のための $1$ を加える。
- $\mathbf{w}$: パラメータベクトル（カテゴリウェイト） ($(d+1) \times 1$)
- $\mathbf{\epsilon}$: 誤差ベクトル ($n \times 1$)

<br>

#### **最適化（最小二乗法）**

予測値 $\hat{y}_i$ と実測値 $y_i$ のズレ（残差）の二乗和を最小化することを目的とします。
これを 目的関数（損失関数） と呼び、$J(\mathbf{w})$ で表します。

$$J(\mathbf{w}) = \sum_{i=1}^n (y_i - \hat{y}_i)^2 = \|\mathbf{y} - \mathbf{Xw}\|^2$$

**正規方程式**:
この目的関数 $J(\mathbf{w})$ を最小にする最適な重み $\mathbf{w}$ は、$\nabla J(\mathbf{w}) = \mathbf{0}$ となる $\mathbf{w}$ を求めることで、以下のように解析的に求まります。

$$\mathbf{w} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

この式により、コンピュータは一発で最適なウェイトを算出します。しかし、この計算を実行するためには、ある重要な数学的条件を満たしている必要があります。

<br>


### **3. 多重共線性と「ダミー変数トラップ」**

ここで、具体的なダミー変数の作成例を見てみましょう。
例えば、「色（赤・青・緑）」という1つの質的変数を考えます。これを数値化するために、各カテゴリに対応する列を作ります。

| ID | 色（Original） | 赤ダミー ($x_1$) | 青ダミー ($x_2$) | 緑ダミー ($x_3$) |
| ----- | ----- | ----- | ----- | ----- |
| 1 | 赤 | 1 | 0 | 0 |
| 2 | 緑 | 0 | 0 | 1 |
| 3 | 青 | 0 | 1 | 0 |

<br>

このように $0/1$ で表現された変数をダミー変数と呼びます。

この例で、すべてのダミー変数（赤・青・緑）を説明変数として使うと、数学的な問題が発生します。

$$\mathbf{x_1} + \mathbf{x_2} + \mathbf{x_3} = \mathbf{1} $$

これは、バイアス項（定数項 $w_0 \times 1$）の列と、ダミー変数の和が完全に相関（線形従属）することを意味します。この状態では行列 $\mathbf{X}^\top \mathbf{X}$ の行列式が $0$ となり、逆行列 $(\mathbf{X}^\top \mathbf{X})^{-1}$ が存在しないため、解が一意に定まりません。これを **ダミー変数トラップ** と呼びます。

**解決策**: $k$ 個のカテゴリがある場合、基準となる1つを除いた $k-1$ 個 のダミー変数を用います。除かれたカテゴリは「基準（ベースライン）」となり、モデルの切片 $w_0$ に吸収されます。

<br>

### **4. 結果の解釈：カテゴリウェイトと重要度（レンジ）**

算出された偏回帰係数 $w$ は、数量化Ⅰ類では **カテゴリウェイト（スコア）** と呼ばれます。
ここでは、係数の具体的な意味と、異なる項目間（例：色と形）の比較方法、およびモデルの評価について定義します。

#### **(1) カテゴリウェイト（基準との差分）**

各カテゴリ $j$ のウェイト $w_j$ は、基準カテゴリ（Baseline）からの予測値 $y$ の変化量 を表します。
基準として削除されたカテゴリのウェイトは便宜上 $0$ とみなします。

- $w_j > 0$: 基準カテゴリに比べて、スコアを押し上げる効果がある。
- $w_j < 0$: 基準カテゴリに比べて、スコアを下げる効果がある。

#### **(2) 異なる項目間の比較：レンジ（Range）による重要度**

「色（Color）」の係数と「形（Shape）」の係数を比較することに意味はあるのでしょうか？
答えは Yes です。なぜなら、すべての係数は 目的変数 $y$（点数や金額など）という共通の単位 で算出されているからです。

各アイテム（項目）が予測にどれだけ強く影響しているか（重要度）を測る指標として、レンジ（範囲） が用いられます。
ある項目 $k$ に属するカテゴリウェイトの最大値を $w_{k,\max}$、最小値を $w_{k,\min}$ とすると、レンジ $R_k$ は以下のように定義されます。
（※基準カテゴリのウェイト $0$ も含めて最大・最小を考えます）

$$R_k = w_{k,\max} - w_{k,\min}$$

**解釈**: レンジ $R_k$ が大きい項目ほど、その項目内での選択（例：何色を選ぶか）が結果 $y$ に与えるインパクトが大きいことを意味します。これにより、「色を変えるべきか、形を変えるべきか」といった施策の優先順位を決定できます。

#### **(3) モデルの評価：決定係数 ($R^2$)**

構築したモデルがどのくらいデータをうまく説明できているかを表す指標として、決定係数（$R^2$） を使います。
$1$ に近いほど当てはまりが良く、$0$ に近いほど説明力が低いことを示します。

$$R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y_i})^2}{\sum_{i=1}^n (y_i - \bar{y})^2} = 1 - \frac{\|\mathbf{y} - \mathbf{\hat{y}}\|^2}{\|\mathbf{y} - \mathbf{\bar{y}}\|^2}$$

- $\mathbf{\hat{y}} = \mathbf{X}\mathbf{\hat{w}}$ : 予測値ベクトル
- $\mathbf{\bar{y}} = \bar{y}\mathbf{1}$ : 全要素が平均値 $\bar{y}$ であるベクトル


---

## **Python 演習課題**

### **mva_05-A**

✅ **目的**: 質的データを行列演算可能な「数値行列（デザイン行列）」に変換するプロセスを理解し、ダミー変数トラップ（多重共線性）の回避方法を確認しましょう。

#### 🖊 **【数理理解】ダミー変数と行列式の手計算**

なぜ「すべてのカテゴリをダミー変数にすると計算できなくなるのか」を、手計算可能な最小の例（2サンプル・2カテゴリ）で確認してみましょう。

**設定**：
- サンプル1：Red
- サンプル2：Blue
- 変数：Bias, Red, Blue

この場合のデザイン行列 $X$ は以下のようになります（$2 \times 3$ 行列）。

$$X =
\begin{pmatrix}
\mathrm{Bias} & \mathrm{Red} & \mathrm{Blue} \\
1 & 1 & 0 \\
1 & 0 & 1
\end{pmatrix}$$

正規方程式で必要となる $\mathbf{X}^\top \mathbf{X}$（$3 \times 3$ 行列）を計算します。

$$\mathbf{X}^\top \mathbf{X} =
\begin{pmatrix}
1 & 1 \\
1 & 0 \\
0 & 1
\end{pmatrix}
\begin{pmatrix}
1 & 1 & 0 \\
1 & 0 & 1
\end{pmatrix}
=
\begin{pmatrix}
2 & 1 & 1 \\
1 & 1 & 0 \\
1 & 0 & 1
\end{pmatrix}$$

この行列の行列式 $\det(\mathbf{X}^\top \mathbf{X})=0$となります。行列式が0の場合、逆行列 $(\mathbf{X}^\top \mathbf{X})^{-1}$ は存在しません。つまり、解（係数）を一意に求めることができません。これがダミー変数トラップの数学的な正体です。

<br>

#### **具体例で確認してみよう！**

上記の現象を Python で確認してみましょう。

In [ ]:
import numpy as np
import pandas as pd

# 1. 小さなデータの定義
# 3サンプル、3カテゴリの単純な例
df_simple = pd.DataFrame({
    'Color': ['Red', 'Blue', 'Green']
})

print("--- 1. 元のデータ ---")
display(df_simple)

# 2. すべてのカテゴリをダミー変数化 (Trapの状態)
X_trap = pd.get_dummies(df_simple['Color'], dtype=int)

# バイアス項(すべて1の列)を追加
X_trap['Bias'] = 1

# 列の並びを整理（Biasを先頭に）
X_trap = X_trap[['Bias', 'Blue', 'Green', 'Red']]

print("\n--- 2. デザイン行列 X (全カテゴリ + バイアス) ---")
display(X_trap)

# 3. 多重共線性(線形従属)の確認
# ダミー変数の和がBiasと等しいかプログラムで確認
sum_dummies = X_trap['Blue'] + X_trap['Green'] + X_trap['Red']
is_dependent = np.all(sum_dummies == X_trap['Bias'])

print(f"\nダミー変数の和 == Bias列 -> {is_dependent}")

# 4. 行列のランク(階数)と行列式(Determinant)の確認
rank = np.linalg.matrix_rank(X_trap.values)
n, d = X_trap.shape

print(f"\n行列のサイズ: {d} 列")
print(f"行列のランク: {rank}")

# 行列式 det(X^T X) を計算
# これが0に近い場合、逆行列は計算できません
XTX = X_trap.T @ X_trap
det = np.linalg.det(XTX)
print(f"det(X^T X): {det:.5f}")

##### **解決策：drop_first=True**

最初のカテゴリ（ここではBlueなど）を除外し、残りの $k-1$ 個の列だけを使うことで、ランク落ちを防ぎます。

In [ ]:
# 5. 最初のカテゴリを除外してダミー変数化 (drop_first=True)
X_proper = pd.get_dummies(df_simple['Color'], drop_first=True, dtype=int)
X_proper['Bias'] = 1

print("\n--- 3. 修正後のデザイン行列 X (k-1カテゴリ + バイアス) ---")
display(X_proper)

# ランクの再確認
rank_proper = np.linalg.matrix_rank(X_proper.values)
n_p, d_p = X_proper.shape

print(f"行列のランク: {rank_proper} (列数: {d_p})")

### **mva_05-B**

✅ **目的**: 実践的なデータセットに対し、``pandas`` での前処理と ``scikit-learn`` による回帰分析を行い、得られた係数を「カテゴリの重要度（スコア）」として解釈・可視化できることを確認しましょう。

### **💻 【実装・応用】 ``scikit-learn`` による数量化Ⅰ類の実装**

実務では、行列計算をゼロから実装するのではなく、最適化されたライブラリを使用します。
数量化Ⅰ類は、数理的には「ダミー変数を用いた重回帰分析」であるため、通常の重回帰分析と同じライブラリ ``scikit-learn`` をそのまま利用して計算できます。

詳細な仕様については、以下の公式ドキュメントを参照してください。
- ``scikit-learn`` 公式ドキュメント: [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

#### **具体的例で確認してみよう！**

ある製品のコンセプト評価データを用いて、どの「デザイン要素」が「評価スコア」にプラス（またはマイナス）の影響を与えているかを分析します。

**データ概要**
- Rating: 製品評価スコア（目的変数 $y$）
- Color: 色（Red, Blue, Green）
- Shape: 形（Round, Square）
- Material: 素材（Wood, Metal, Plastic）

In [ ]:
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

# 1. データの生成 (サンプルデータ)
data = {
    'Rating': [3.5, 4.2, 2.8, 5.0, 3.1, 4.5, 3.8, 2.5, 4.0, 3.2],
    'Color': ['Red', 'Blue', 'Green', 'Red', 'Green', 'Blue', 'Red', 'Green', 'Blue', 'Red'],
    'Shape': ['Round', 'Square', 'Square', 'Round', 'Round', 'Square', 'Square', 'Round', 'Round', 'Square'],
    'Material': ['Plastic', 'Metal', 'Wood', 'Metal', 'Wood', 'Metal', 'Plastic', 'Wood', 'Plastic', 'Wood']
}
df_product = pd.DataFrame(data)

print("--- 分析データ ---")
display(df_product.head())

# 2. ダミー変数化 (One-Hot Encoding)
# drop_first=True で各項目の最初のカテゴリを基準(Baseline)にします
X = pd.get_dummies(df_product[['Color', 'Shape', 'Material']], drop_first=True, dtype=int)
y = df_product['Rating']

print("\n--- 説明変数(X)の列名 ---")
print(X.columns.tolist())

# 3. モデルの学習
model = LinearRegression()
model.fit(X, y)

# 4. 結果の解釈 (カテゴリウェイト)
# 偏回帰係数を取得
weights = pd.Series(model.coef_, index=X.columns)
intercept = model.intercept_

print(f"\nベースライン(切片): {intercept:.4f}")
print("\nカテゴリウェイト (基準カテゴリからの増減):")
print(weights.sort_values(ascending=False))

**可視化：カテゴリごとの影響度（係数プロット）**

係数の大きさを棒グラフで可視化します。
- 正の値: 基準カテゴリより評価が高い
- 負の値: 基準カテゴリより評価が低い
- 0 (表示なし): 基準カテゴリ（Baseline）

※本来の数量化Ⅰ類では、基準カテゴリも含めてウェイトを調整（中心化）することもありますが、ここでは「基準との比較」という回帰分析の解釈で可視化します。

In [ ]:
# 5. 係数の可視化
plt.figure(figsize=(10, 6))
weights.sort_values().plot(kind='barh', color='teal')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.title("Category Weights (Impact on Rating)")
plt.xlabel("Coefficient Value (Difference from Baseline)")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

**項目の重要度（レンジ）の比較**

どの要素（色、形、素材）が製品評価に最も影響を与えているかを判断するため、各項目の レンジ（範囲 = 最大値 - 最小値） を計算して比較します。
※基準カテゴリ（係数0）も考慮に入れて計算します。

In [ ]:
# 6. 項目の重要度(レンジ)の計算
items = ['Color', 'Shape', 'Material']
ranges = {}

print("\n--- 項目の重要度 (Range) ---")

for item in items:
    # その項目に含まれるダミー変数の係数を抽出
    # 例: "Color_" で始まる列の係数
    item_weights = [weights[col] for col in X.columns if col.startswith(item)]

    # 基準カテゴリ(係数0)を追加して、最大値と最小値を比較
    item_weights.append(0)

    # レンジの計算
    r = max(item_weights) - min(item_weights)
    ranges[item] = r
    print(f"{item}: {r:.4f}")

# 可視化
plt.figure(figsize=(8, 4))
pd.Series(ranges).sort_values().plot(kind='barh', color='salmon')
plt.title("Importance of Each Item (Range)")
plt.xlabel("Range (Max - Min of Weights)")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()